In [1]:
import requests

url = "https://geodesy.unr.edu/vlm/VLM_Global_Imaged.txt"
r = requests.get(url)
r.raise_for_status()  # Ensure the request was successful

# Display the first few lines
print("\n".join(r.text.splitlines()[:10]))

# Save to local file
with open("VLM_Global_Imaged.txt", "w") as f:
    f.write(r.text)

print("Downloaded interpolated VLM text file.")


 -180.00 -90.00    0.782    0.348    0.975
 -179.75 -90.00    0.782    0.348    0.975
 -179.50 -90.00    0.782    0.348    0.975
 -179.25 -90.00    0.782    0.348    0.975
 -179.00 -90.00    0.782    0.348    0.975
 -178.75 -90.00    0.782    0.348    0.975
 -178.50 -90.00    0.782    0.348    0.975
 -178.25 -90.00    0.782    0.348    0.975
 -178.00 -90.00    0.782    0.348    0.975
 -177.75 -90.00    0.782    0.348    0.975
Downloaded interpolated VLM text file.


In [2]:
import requests
from pathlib import Path

# ------------ USER INPUT ------------
# Bounding box in lon/lat (WGS84)
MIN_LON, MIN_LAT = -98.8, 25.0   # left,  bottom
MAX_LON, MAX_LAT = -93.0, 30.6   # right, top

SRC_URL   = "https://geodesy.unr.edu/vlm/VLM_Global_Imaged.txt"
OUT_TXT   = Path("VLM_bbox.txt")   # CSV-style TXT: lon,lat,vlm_mm_per_yr
# -----------------------------------

def in_bbox(lon, lat):
    return (MIN_LON <= lon <= MAX_LON) and (MIN_LAT <= lat <= MAX_LAT)

with requests.get(SRC_URL, stream=True, timeout=300) as r, OUT_TXT.open("w", encoding="utf-8") as out:
    r.raise_for_status()
    out.write("lon,lat,vlm_mm_per_yr\n")
    for raw in r.iter_lines(decode_unicode=True):
        if not raw or raw.startswith("#"):
            continue
        parts = raw.split()
        if len(parts) < 3:
            continue
        # Most NGL text grids use: lon lat value
        lon, lat, val = map(float, parts[:3])

        # If longitudes are 0..360, normalize to -180..180 (just in case)
        if lon > 180.0:
            lon -= 360.0

        if in_bbox(lon, lat):
            out.write(f"{lon:.6f},{lat:.6f},{val:.6f}\n")

print(f"Saved subset to: {OUT_TXT.resolve()}")


Saved subset to: C:\Users\sahad2\VLM_bbox.txt


In [3]:
import requests
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
from pathlib import Path

# ---------- USER INPUT ----------
shp_zip = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"
src_url = "https://geodesy.unr.edu/vlm/VLM_Global_Imaged.txt"
out_csv = Path("VLM_clipped_polygon.csv")
# --------------------------------

# 1. Load polygon from zipped shapefile
gdf = gpd.read_file(shp_zip)

# 2. Ensure CRS is EPSG:32614 (UTM 14N) -> reproject to EPSG:4326 (WGS84 lon/lat)
if gdf.crs is None:
    gdf.set_crs("EPSG:32614", inplace=True)
gdf = gdf.to_crs("EPSG:4326")

# Union into a single polygon (if multiple parts)
poly = gdf.unary_union

# 3. Stream download the TXT and filter
rows = []
with requests.get(src_url, stream=True, timeout=300) as r:
    r.raise_for_status()
    for raw in r.iter_lines(decode_unicode=True):
        if not raw or raw.startswith("#"):
            continue
        parts = raw.split()
        if len(parts) < 3:
            continue
        lon, lat, val = map(float, parts[:3])
        # Convert longitude if 0–360 to -180–180
        if lon > 180:
            lon -= 360
        pt = Point(lon, lat)
        if poly.contains(pt):
            rows.append((lon, lat, val))

# 4. Save to CSV (lon, lat, vlm_mm_per_yr)
df = pd.DataFrame(rows, columns=["lon", "lat", "vlm_mm_per_yr"])
df.to_csv(out_csv, index=False)

print(f"Saved {len(df)} points inside polygon to {out_csv.resolve()}")


C:\Users\sahad2\AppData\Local\Temp\ipykernel_2612\3105198773.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  poly = gdf.unary_union


Saved 19 points inside polygon to C:\Users\sahad2\VLM_clipped_polygon.csv
